In this notebook we will clean and filter bindinding_db and biolip
and then we will merge them into a single dataset.



In [39]:
import pandas as pd

# === SETTINGS ===
bindingdb_file = "BindingDB_All.tsv"    
output_file = "bindingdb_filtered.csv"  # final saved dataset

# Columns we actually care about
needed_cols = [
    "Ligand SMILES",
    "UniProt (SwissProt) Primary ID of Target Chain 1",
    "Ki (nM)", "IC50 (nM)", "Kd (nM)", "EC50 (nM)"
]

# Storage for chunk results
filtered_chunks = []

# === PROCESS FILE IN CHUNKS ===
chunk_iter = pd.read_csv(
    bindingdb_file,
    sep="\t",
    usecols=lambda c: c in needed_cols,
    chunksize=50_000,   # adjust based on RAM
    low_memory=True
)

for chunk in chunk_iter:
    # Keep rows where at least one bioactivity value is present
    chunk = chunk.dropna(subset=["Ki (nM)", "IC50 (nM)", "Kd (nM)", "EC50 (nM)"], how="all")

    filtered_chunks.append(chunk)

# === COMBINE ALL CHUNKS ===
final_df = pd.concat(filtered_chunks, ignore_index=True)

# === SAVE TO DISK ===
final_df.to_csv(output_file, index=False)

print("Done! Final dataset shape:", final_df.shape)


/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_13360/2955721277.py:26: DtypeWarning: Columns (8,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunk_iter:
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_13360/2955721277.py:26: DtypeWarning: Columns (8,9,10,11,44) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunk_iter:
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_13360/2955721277.py:26: DtypeWarning: Columns (8,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunk_iter:
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_13360/2955721277.py:26: DtypeWarning: Columns (8,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunk_iter:
/var/folders/48/r_dz_rls5y7flr7cw8l6z6s00000gn/T/ipykernel_13360/2955721277.py:26: DtypeWarning: Columns (8,9,10,11) have mixed

Done! Final dataset shape: (3075963, 6)


In [40]:
final_df.shape

(3075963, 6)

In [41]:
import pandas as pd

# === SETTINGS ===
biolip_file = "biolip.txt"              
output_file = "biolip_cleaned.csv"      # final cleaned file

# === HEADER for BioLiP TXT ===
header = [
    "PDB_ID",
    "Chain",
    "Kd_or_Score",
    "Binding_Site_ID",
    "Ligand_ID",
    "Ligand_Chain",
    "Num_Residues",
    "Binding_Residues",
    "Binding_Residues_Alt",
    "EC_numbers",
    "GO_terms",
    "Unused1",
    "Unused2",
    "UniProt_ID",
    "Unused3",
    "Sequence_Length",
    "Sequence"
]

# === READ BIOlip FILE (no header) ===
df = pd.read_csv(
    biolip_file,
    sep="\t",
    names=header,
    dtype=str,             # avoid type issues
    low_memory=False
)

# === SELECT ONLY IMPORTANT COLUMNS ===
important_cols = [
    "Ligand_ID",
    "UniProt_ID",
    "Kd_or_Score"
]

bio_clean = df[important_cols].copy()

# === SAVE RESULT ===
bio_clean.to_csv(output_file, index=False)

bio_clean.head()


,,,,Ligand_ID,UniProt_ID,Kd_or_Score
101m,A,2.07,BS01,F44 R46 V69 S93 H94 H98 I100 Y104,P02185,1
102m,A,1.84,BS01,F44 R46 T68 L90 S93 H94 H98 I100 Y104,P02185,1
103m,A,2.07,BS01,F44 R46 S93 H94 H98 I100 Y104,P02185,1
104m,A,1.71,BS01,F43 R45 V68 S92 H93 H97 I99 Y103 F138,P02185,1
105m,A,2.02,BS01,F43 R45 H64 V68 L89 H93 H97 I99,P02185,1


In [42]:
bio_clean.columns

Index(['Ligand_ID', 'UniProt_ID', 'Kd_or_Score'], dtype='object')

In [43]:
import pandas as pd
import numpy as np

# ---- Load extracted bindingdb dataset ----
df = pd.read_csv("bindingdb_filtered.csv")

# ---- Convert activity columns to numeric ----
activity_cols = ["Ki (nM)", "IC50 (nM)", "Kd (nM)", "EC50 (nM)"]

for col in activity_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- Create a single affinity value (pick the first available) ----
df["Affinity"] = df[activity_cols].bfill(axis=1).iloc[:, 0]

# ---- Keep ONLY rows where affinity > 10,000 nM (10 µM) ----
negatives = df[df["Affinity"] > 10000]

# ---- Keep only essential columns ----
negatives = negatives[["Ligand SMILES", 
                       "UniProt (SwissProt) Primary ID of Target Chain 1",
                       "Affinity"]]

# ---- Save clean negative dataset ----
negatives.to_csv("bindingdb_negatives_filtered.csv", index=False)

negatives.head(), negatives.shape


(                                         Ligand SMILES  \
 123  CC(=O)NCCN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](O)[C...   
 124  O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCNC(=O)Cc...   
 125  O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCn2ccn...   
 126  O[C@@H]1[C@@H](O)[C@@H](Cc2ccccc2)N(CCCCCN2CCO...   
 127  CC(C)(C)C(O)CN1[C@H](Cc2ccccc2)[C@H](O)[C@@H](...   
 
     UniProt (SwissProt) Primary ID of Target Chain 1  Affinity  
 123                                           P03367   12500.0  
 124                                           P03367   12500.0  
 125                                           P03367   12500.0  
 126                                           P03367   12500.0  
 127                                           P03367   21000.0  ,
 (278822, 3))

In [48]:
bindingdb_neg.columns
bindingdb_neg.shape

(278822, 4)

In [45]:
import pandas as pd


bindingdb_neg = pd.read_csv("bindingdb_negatives_filtered.csv")
biolip_clean = pd.read_csv("biolip_cleaned.csv")

# ---- Standardize column names ----
# We rename to a common format: (Drug, Target, Affinity)
bindingdb_neg = bindingdb_neg.rename(columns={
    "Ligand SMILES": "Drug",
    "UniProt (SwissProt) Primary ID of Target Chain 1": "Target",
    "Affinity": "Affinity"
})

biolip_clean = biolip_clean.rename(columns={
    "Ligand_ID": "Drug",
    "UniProt_ID": "Target",
    "Kd_or_Score": "Affinity"
})

# ---- Add dataset source labels (optional but useful) ----
bindingdb_neg["Source"] = "BindingDB"
biolip_clean["Source"] = "BioLiP"

# ---- Select the same columns ----
bindingdb_neg = bindingdb_neg[["Drug", 
                       "Target",
                       "Affinity", "Source"]]
biolip_clean = biolip_clean[["Drug", "Target", "Affinity", "Source"]]

# ---- Merge them (stack the rows) ----
merged_neg = pd.concat([bindingdb_neg, biolip_clean], ignore_index=True)

# ---- Remove duplicates ----
merged_neg = merged_neg.drop_duplicates(subset=["Drug", "Target"])

# ---- Save final combined dataset ----
merged_neg.to_csv("negative_dti_merged.csv", index=False)

merged_neg.head()
merged_neg.shape


(853716, 4)

In [47]:
#check how many of the rows have source bindingdb and how many biolip
merged_neg['Source'].value_counts()


Source
BioLiP       615203
BindingDB    238513
Name: count, dtype: int64